# YOLOv8 Instance Segmentation for Retinal Structures

This notebook uses Ultralytics YOLOv8 to perform instance segmentation on fundus images, identifying and isolating important retinal structures and lesions for diabetic retinopathy diagnosis.

## Key Concepts

**YOLOv8 Segmentation** combines:
- **Detection**: Finds bounding boxes of objects
- **Segmentation**: Generates precise masks for each detected object
- **Real-time performance**: Optimized for speed

For retinal images, we can use pre-trained models or fine-tune on custom retinal datasets to detect:
- Optic disc
- Blood vessels
- Microaneurysms
- Hemorrhages
- Hard exudates
- Cotton-wool spots

In [ ]:
# Install YOLOv8 if needed
import subprocess
import sys

try:
    import ultralytics
    print(f"Ultralytics already installed: {ultralytics.__version__}")
except ImportError:
    print("Installing ultralytics...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
    print("Installation complete!")

In [ ]:
# Import necessary libraries
import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Polygon
import os
import warnings
warnings.filterwarnings('ignore')

from ultralytics import YOLO
from pathlib import Path

# Configure matplotlib
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['image.cmap'] = 'gray'

print("Libraries imported successfully!")

## 1. Load YOLOv8 Model and Prepare Data

In [ ]:
# Load pre-trained YOLOv8 segmentation model
# Available sizes: nano (n), small (s), medium (m), large (l), xlarge (x)
# We use 'medium' as a good balance between speed and accuracy

print("Loading YOLOv8 model (this may take a moment)...")
model = YOLO('yolov8m-seg.pt')  # 'seg' suffix indicates segmentation model
print(f"Model loaded: {model.model_name if hasattr(model, 'model_name') else 'YOLOv8m-seg'}")
print(f"Model info:\n{model.info()}")

In [ ]:
# Helper function to find images
def find_images(base_path, extensions=['.jpeg', '.jpg', '.png']):
    """Find all images in a directory and subdirectories"""
    images = []
    for root, dirs, files in os.walk(base_path):
        for file in files:
            if any(file.lower().endswith(ext) for ext in extensions):
                images.append(os.path.join(root, file))
    return sorted(images)

# Load sample images from dataset
data_path = 'data/1-mild'
images = find_images(data_path)

print(f"Found {len(images)} images in {data_path}")
if images:
    print(f"First image: {images[0]}")

## 2. Run Segmentation on Single Image

In [ ]:
# Load and segment a single image
if len(images) > 0:
    img_path = images[10]  # Select a sample image
    print(f"Processing: {img_path}")
    
    # Read image
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    print(f"Image shape: {img_rgb.shape}")
    
    # Run YOLOv8 segmentation
    # conf: confidence threshold (0-1), lower = more detections
    # iou: IoU threshold for NMS (0-1)
    results = model.predict(
        source=img_path,
        conf=0.25,  # Confidence threshold
        iou=0.45,   # NMS IoU threshold
        imgsz=640,  # Input image size
        device=0 if torch.cuda.is_available() else 'cpu',
        verbose=False
    )
    
    result = results[0]
    print(f"\nDetections: {len(result.boxes)}")
    
    if result.masks is not None:
        print(f"Masks shape: {result.masks.data.shape}")
        print(f"Confidence scores: {result.boxes.conf}")
        print(f"Class IDs: {result.boxes.cls}")
    else:
        print("No masks detected")
else:
    print("No images found")

In [ ]:
import torch

# Visualize detections and segmentations
if len(images) > 0 and result is not None:
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    
    # Original image
    axes[0].imshow(img_rgb)
    axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
    axes[0].axis('off')
    
    # Image with bounding boxes
    img_boxes = img_rgb.copy()
    if result.boxes is not None:
        for box, conf, cls_id in zip(result.boxes.xyxy, result.boxes.conf, result.boxes.cls):
            x1, y1, x2, y2 = map(int, box.tolist())
            cv2.rectangle(img_boxes, (x1, y1), (x2, y2), (0, 255, 0), 3)
            cv2.putText(img_boxes, f'{conf:.2f}', (x1, y1-10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    
    axes[1].imshow(img_boxes)
    axes[1].set_title(f'Detections (boxes): {len(result.boxes)} objects', fontsize=14, fontweight='bold')
    axes[1].axis('off')
    
    # Segmentation masks
    img_seg = img_rgb.copy().astype(float)
    
    if result.masks is not None:
        masks = result.masks.data.cpu().numpy()
        h, w = img_rgb.shape[:2]
        
        # Create colored overlay for each mask
        colors = plt.cm.hsv(np.linspace(0, 1, len(masks)))
        
        for idx, (mask, conf, cls_id) in enumerate(zip(masks, result.boxes.conf, result.boxes.cls)):
            # Resize mask to image size
            mask_resized = cv2.resize(mask, (w, h), interpolation=cv2.INTER_LINEAR)
            mask_resized = (mask_resized > 0.5).astype(np.uint8)
            
            # Apply color overlay
            color = np.array(colors[idx][:3]) * 255
            img_seg[mask_resized > 0] = img_seg[mask_resized > 0] * 0.5 + color * 0.5
    
    img_seg = np.clip(img_seg, 0, 255).astype(np.uint8)
    axes[2].imshow(img_seg)
    axes[2].set_title(f'Instance Segmentation: {len(masks) if result.masks is not None else 0} masks',
                     fontsize=14, fontweight='bold')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

## 3. Extract and Analyze Individual Masks

In [ ]:
def extract_mask_properties(result, img_rgb):
    """
    Extract detailed properties from each detected mask.
    
    Returns:
        List of dictionaries with mask properties
    """
    mask_properties = []
    
    if result.masks is None:
        return mask_properties
    
    masks = result.masks.data.cpu().numpy()
    h, w = img_rgb.shape[:2]
    
    for idx, (mask, box, conf, cls_id) in enumerate(zip(
        masks, result.boxes.xyxy, result.boxes.conf, result.boxes.cls
    )):
        # Resize mask to image dimensions
        mask_resized = cv2.resize(mask, (w, h), interpolation=cv2.INTER_LINEAR)
        mask_binary = (mask_resized > 0.5).astype(np.uint8)
        
        # Calculate properties
        area = mask_binary.sum()
        x1, y1, x2, y2 = map(int, box.tolist())
        bbox_area = (x2 - x1) * (y2 - y1)
        fill_ratio = area / (bbox_area + 1e-6)
        
        # Get bounding box contour
        contours, _ = cv2.findContours(mask_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        # Extract color statistics inside mask
        masked_pixels = img_rgb[mask_binary > 0]
        if len(masked_pixels) > 0:
            mean_color = masked_pixels.mean(axis=0)
            std_color = masked_pixels.std(axis=0)
        else:
            mean_color = np.array([0, 0, 0])
            std_color = np.array([0, 0, 0])
        
        mask_properties.append({
            'id': idx,
            'confidence': conf.item(),
            'class_id': cls_id.item(),
            'mask': mask_binary,
            'bbox': (x1, y1, x2, y2),
            'area': area,
            'bbox_area': bbox_area,
            'fill_ratio': fill_ratio,
            'mean_color': mean_color,
            'std_color': std_color,
            'num_contours': len(contours)
        })
    
    return mask_properties

# Extract properties
if result is not None and result.masks is not None:
    props = extract_mask_properties(result, img_rgb)
    
    print(f"\nDetailed Mask Analysis ({len(props)} objects):\n")
    print(f"{'ID':<3} {'Conf':<6} {'Area':<8} {'Fill%':<7} {'Mean R':<7} {'Mean G':<7} {'Mean B':<7}")
    print("-" * 56)
    
    for prop in props:
        print(f"{prop['id']:<3} {prop['confidence']:<6.2f} {prop['area']:<8} "
              f"{prop['fill_ratio']*100:<7.1f} {prop['mean_color'][0]:<7.1f} "
              f"{prop['mean_color'][1]:<7.1f} {prop['mean_color'][2]:<7.1f}")

## 4. Create Binary and Composite Masks

In [ ]:
# Create different mask representations
if result is not None and result.masks is not None and len(props) > 0:
    # Create combined mask showing all detections
    combined_mask = np.zeros((img_rgb.shape[0], img_rgb.shape[1]), dtype=np.uint8)
    mask_colored = np.zeros_like(img_rgb)
    
    colors = plt.cm.tab20(np.linspace(0, 1, len(props)))
    
    for prop, color in zip(props, colors):
        combined_mask[prop['mask'] > 0] = prop['id'] + 1
        mask_colored[prop['mask'] > 0] = np.array(color[:3]) * 255
    
    # Visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Original
    axes[0, 0].imshow(img_rgb)
    axes[0, 0].set_title('Original Image', fontweight='bold')
    axes[0, 0].axis('off')
    
    # Binary combined mask
    axes[0, 1].imshow(combined_mask > 0, cmap='gray')
    axes[0, 1].set_title('Binary Mask (All Detections)', fontweight='bold')
    axes[0, 1].axis('off')
    
    # Instance segmentation (colored)
    axes[0, 2].imshow(mask_colored.astype(np.uint8))
    axes[0, 2].set_title('Instance Segmentation (All Masks)', fontweight='bold')
    axes[0, 2].axis('off')
    
    # Show largest masks individually
    props_sorted = sorted(props, key=lambda x: x['area'], reverse=True)
    
    for idx in range(min(3, len(props_sorted))):
        row, col = 1, idx
        prop = props_sorted[idx]
        mask_display = prop['mask'] * 255
        
        axes[row, col].imshow(mask_display, cmap='gray')
        axes[row, col].set_title(
            f"Mask #{prop['id']+1} (Conf: {prop['confidence']:.2f})\n"
            f"Area: {prop['area']} pixels",
            fontweight='bold', fontsize=10
        )
        axes[row, col].axis('off')
    
    plt.tight_layout()
    plt.show()

## 5. Process Multiple Images

In [ ]:
# Process multiple images
sample_indices = [5, 10, 15, 20, 25, 30]  # Select 6 sample images
sample_images = [images[i] for i in sample_indices if i < len(images)]

print(f"Processing {len(sample_images)} sample images...")

segmentation_results = []

for img_path in sample_images:
    results = model.predict(
        source=img_path,
        conf=0.25,
        iou=0.45,
        imgsz=640,
        device=0 if torch.cuda.is_available() else 'cpu',
        verbose=False
    )
    
    result = results[0]
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    num_detections = len(result.boxes) if result.boxes is not None else 0
    
    segmentation_results.append({
        'path': img_path,
        'image': img_rgb,
        'result': result,
        'num_detections': num_detections
    })
    
    print(f"  {Path(img_path).name}: {num_detections} objects detected")

print("Processing complete!")

In [ ]:
# Visualize multiple images with segmentations
fig, axes = plt.subplots(2, 3, figsize=(22, 14))
axes = axes.flatten()

for ax_idx, seg_result in enumerate(segmentation_results[:6]):
    img = seg_result['image']
    result = seg_result['result']
    
    # Create segmentation overlay
    img_overlay = img.copy().astype(float)
    
    if result.masks is not None:
        masks = result.masks.data.cpu().numpy()
        h, w = img.shape[:2]
        colors = plt.cm.hsv(np.linspace(0, 1, len(masks)))
        
        for mask, color in zip(masks, colors):
            mask_resized = cv2.resize(mask, (w, h), interpolation=cv2.INTER_LINEAR)
            mask_binary = (mask_resized > 0.5).astype(np.uint8)
            color_rgb = np.array(color[:3]) * 255
            img_overlay[mask_binary > 0] = img_overlay[mask_binary > 0] * 0.4 + color_rgb * 0.6
    
    img_overlay = np.clip(img_overlay, 0, 255).astype(np.uint8)
    
    axes[ax_idx].imshow(img_overlay)
    axes[ax_idx].set_title(
        f"{Path(seg_result['path']).name}\n"
        f"Detections: {seg_result['num_detections']}",
        fontweight='bold'
    )
    axes[ax_idx].axis('off')

plt.suptitle('YOLOv8 Segmentation Results on Multiple Images', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Extract Segmentation Features for Classifier

In [ ]:
def create_segmentation_features(img_rgb, result):
    """
    Extract features from YOLOv8 segmentation for use in classifier.
    
    Returns:
        Dictionary with segmentation-based features
    """
    features = {
        'num_objects': 0,
        'total_mask_area': 0,
        'mask_coverage': 0,
        'max_confidence': 0,
        'mean_confidence': 0,
        'mask_diversity': 0,  # Variation in object sizes
        'largest_object_area': 0,
        'object_sizes': []
    }
    
    if result.boxes is None or result.masks is None:
        return features
    
    masks = result.masks.data.cpu().numpy()
    h, w = img_rgb.shape[:2]
    total_pixels = h * w
    
    features['num_objects'] = len(masks)
    features['max_confidence'] = result.boxes.conf.max().item()
    features['mean_confidence'] = result.boxes.conf.mean().item()
    
    sizes = []
    total_area = 0
    
    for mask in masks:
        mask_resized = cv2.resize(mask, (w, h), interpolation=cv2.INTER_LINEAR)
        mask_binary = (mask_resized > 0.5).astype(np.uint8)
        area = mask_binary.sum()
        sizes.append(area)
        total_area += area
    
    features['total_mask_area'] = total_area
    features['mask_coverage'] = total_area / total_pixels
    features['object_sizes'] = sizes
    
    if sizes:
        features['largest_object_area'] = max(sizes)
        features['mask_diversity'] = np.std(sizes) / (np.mean(sizes) + 1e-6)
    
    return features

# Extract features from results
print("Segmentation Features Summary:")
print("=" * 80)
print(f"{'Image':<30} {'Objects':<10} {'Coverage%':<12} {'Mean Conf':<12}")
print("-" * 80)

all_features = []

for seg_result in segmentation_results:
    features = create_segmentation_features(seg_result['image'], seg_result['result'])
    all_features.append(features)
    
    img_name = Path(seg_result['path']).name
    print(f"{img_name:<30} {features['num_objects']:<10} "
          f"{features['mask_coverage']*100:<12.2f} {features['mean_confidence']:<12.3f}")

print("=" * 80)

## 7. Save Segmentation Masks

In [ ]:
# Create output directory for masks
output_dir = Path('segmentation_masks')
output_dir.mkdir(exist_ok=True)

print(f"Saving segmentation masks to {output_dir}/")

for seg_result in segmentation_results:
    img_path = seg_result['path']
    result = seg_result['result']
    img = seg_result['image']
    
    if result.masks is not None:
        masks = result.masks.data.cpu().numpy()
        h, w = img.shape[:2]
        
        # Create combined mask
        combined_mask = np.zeros((h, w), dtype=np.uint8)
        for idx, mask in enumerate(masks):
            mask_resized = cv2.resize(mask, (w, h), interpolation=cv2.INTER_LINEAR)
            combined_mask[mask_resized > 0.5] = idx + 1
        
        # Save mask
        img_stem = Path(img_path).stem
        mask_path = output_dir / f"{img_stem}_mask.png"
        cv2.imwrite(str(mask_path), combined_mask)
        print(f"  Saved: {mask_path}")

print("All masks saved!")

## 8. Fine-tuning on Custom Dataset (Optional)

If you have labeled data with masks, you can fine-tune YOLOv8 on your specific retinal structures.

```python
# Fine-tune on custom dataset
from ultralytics import YOLO

model = YOLO('yolov8m-seg.pt')

# Train on custom dataset (in COCO format)
results = model.train(
    data='path/to/retinal_dataset.yaml',  # Dataset YAML file
    epochs=100,
    imgsz=640,
    device=0,  # GPU device
    batch=16,
    patience=20,
    save=True,
    project='retinal_segmentation',
    name='yolov8m_custom'
)

# Use fine-tuned model
model = YOLO('runs/detect/yolov8m_custom/weights/best.pt')
results = model.predict(source='path/to/image.jpg')
```

## 9. Integration with Classifier

Use segmentation masks as preprocessing for the diabetic retinopathy classifier:

```python
def apply_yolo_preprocessing(img_path):
    """Apply YOLOv8 segmentation and use masks for preprocessing."""
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    
    # Run segmentation
    results = model.predict(source=img_path, conf=0.25, verbose=False)
    result = results[0]
    
    # Create mask for important regions
    if result.masks is not None:
        h, w = img_rgb.shape[:2]
        enhanced_img = img_rgb.copy().astype(float)
        
        masks = result.masks.data.cpu().numpy()
        for mask in masks:
            mask_resized = cv2.resize(mask, (w, h), interpolation=cv2.INTER_LINEAR)
            # Enhance regions with detections
            enhanced_img[mask_resized > 0.5] *= 1.2
        
        enhanced_img = np.clip(enhanced_img, 0, 255).astype(np.uint8)
        return enhanced_img
    
    return img_rgb
```

## Summary

### YOLOv8 Segmentation Advantages:
- **Fast**: Real-time inference
- **Accurate**: Instance segmentation with high precision
- **Flexible**: Can be fine-tuned on custom retinal datasets
- **Portable**: Exportable to multiple formats (ONNX, TensorRT, CoreML)

### Key Parameters to Adjust:
- `conf`: Confidence threshold (lower = more detections, but more false positives)
- `iou`: NMS IoU threshold (controls overlap between boxes)
- `imgsz`: Input image size (larger = more detail, slower)

### Next Steps:
1. Collect labeled retinal images with mask annotations
2. Create a COCO-formatted dataset
3. Fine-tune YOLOv8 on your specific retinal structures
4. Integrate segmentation masks into your classifier pipeline